# DesertMap Random Forest Primary Model

This notebook trains the Random Forest primary model for `LILATracts_1And10` and compares it against the Logistic Regression baseline. It also pre-computes tract-level predictions and SHAP explanations for the Streamlit app.

## Section 0: Setup

This section loads the cleaned modeling data, removes the tract identifier for training, and loads baseline metrics for comparison.

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    average_precision_score,
    make_scorer,
    f1_score,
)
import shap

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data/processed/modeling_data_clean.csv'
BASELINE_METRICS_PATH = PROJECT_ROOT / 'outputs/baseline_metrics.json'
FIGURE_DIR = PROJECT_ROOT / 'outputs/figures'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FEATURES = [
    'MedianFamilyIncome',
    'PovertyRate',
    'pct_nhblack10',
    'pct_hisp10',
    'pct_nhwhite10',
    'pct_hunv10',
    'pct_snap16',
    'Pop2010',
    'Urban',
]
TARGET_COL = 'LILATracts_1And10'
FIPS_CANDIDATES = ['CensusTract', 'TractFIPS', 'FIPS']
DEMOGRAPHIC_FEATURES = ['pct_nhblack10', 'pct_hisp10', 'pct_nhwhite10']

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing required data file: {DATA_PATH}')
if not BASELINE_METRICS_PATH.exists():
    raise FileNotFoundError(f'Missing baseline metrics file: {BASELINE_METRICS_PATH}')

modeling_data_full = pd.read_csv(DATA_PATH)

missing_required_columns = [col for col in MODEL_FEATURES + [TARGET_COL] if col not in modeling_data_full.columns]
if missing_required_columns:
    raise KeyError(f'Missing required columns: {missing_required_columns}')

fips_col = next((col for col in FIPS_CANDIDATES if col in modeling_data_full.columns), None)
if fips_col is None:
    raise KeyError(f'No FIPS identifier column found. Tried: {FIPS_CANDIDATES}')

with open(BASELINE_METRICS_PATH) as metrics_file:
    baseline_metrics = json.load(metrics_file)

modeling_data = modeling_data_full.drop(columns=[fips_col])

print(f'Dataset shape after dropping {fips_col}: {modeling_data.shape[0]:,} rows x {modeling_data.shape[1]:,} columns')
print('Class balance:')
print(modeling_data[TARGET_COL].value_counts(normalize=True).rename('proportion').to_frame())

## Section 1: Train/Test Split

The split matches notebook 03 exactly: 80/20, stratified on the target, with `random_state=42`. Features are not scaled because Random Forests split on feature thresholds and do not require standardized inputs.

In [ ]:
X = modeling_data[MODEL_FEATURES]
y = modeling_data[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f'Train size: {X_train.shape[0]:,} rows')
print(f'Test size: {X_test.shape[0]:,} rows')
print('\nTrain class balance:')
print(y_train.value_counts(normalize=True).rename('proportion').to_frame())
print('\nTest class balance:')
print(y_test.value_counts(normalize=True).rename('proportion').to_frame())

## Section 2: Hyperparameter Tuning with GridSearchCV

Grid search uses stratified folds and optimizes F1 for the food desert class. Class weights remain balanced throughout tuning so the minority class is not ignored.

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced'],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = make_scorer(f1_score, pos_label=1)

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    verbose=1,
)

start_time = time.perf_counter()
grid_search.fit(X_train, y_train)
tuning_time = time.perf_counter() - start_time

print(f'Tuning time: {tuning_time:.3f} seconds')
print(f'Best params: {grid_search.best_params_}')
print(f'Best CV F1 score: {grid_search.best_score_:.3f}')

best_rf = grid_search.best_estimator_

## Section 3: Model Evaluation

Evaluation uses the held-out test set created with the same split as the baseline notebook.

### 3a. Classification Report

The report summarizes threshold-based performance, while ROC-AUC and Average Precision evaluate ranking quality across thresholds.

In [ ]:
y_pred = best_rf.predict(X_test)
y_score = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not food desert', 'Food desert']))

roc_auc = roc_auc_score(y_test, y_score)
average_precision = average_precision_score(y_test, y_score)

print(f'ROC-AUC: {roc_auc:.3f}')
print(f'Average Precision: {average_precision:.3f}')

### 3b. Model Comparison Table

The comparison table uses the same metric names as the Logistic Regression baseline. The winner column marks the stronger model for each metric.

In [ ]:
with open(BASELINE_METRICS_PATH) as metrics_file:
    baseline_metrics = json.load(metrics_file)

report = classification_report(
    y_test,
    y_pred,
    target_names=['Not food desert', 'Food desert'],
    output_dict=True,
)

rf_metrics = {
    'Accuracy': report['accuracy'],
    'Precision (food desert class)': report['Food desert']['precision'],
    'Recall (food desert class)': report['Food desert']['recall'],
    'F1 (food desert class)': report['Food desert']['f1-score'],
    'ROC-AUC': roc_auc,
    'Average Precision': average_precision,
}

comparison_rows = []
for metric in rf_metrics:
    logistic_value = baseline_metrics[metric]
    rf_value = rf_metrics[metric]
    if np.isclose(logistic_value, rf_value):
        winner = 'Tie'
    elif rf_value > logistic_value:
        winner = 'Random Forest'
    else:
        winner = 'Logistic Regression'
    comparison_rows.append({
        'Metric': metric,
        'Logistic Regression': logistic_value,
        'Random Forest': rf_value,
        'Winner': winner,
    })

comparison_table = pd.DataFrame(comparison_rows)

def highlight_winner(row):
    styles = [''] * len(row)
    if row['Winner'] == 'Logistic Regression':
        styles[row.index.get_loc('Logistic Regression')] = 'font-weight: bold; background-color: #DDEFE8'
    elif row['Winner'] == 'Random Forest':
        styles[row.index.get_loc('Random Forest')] = 'font-weight: bold; background-color: #DDEFE8'
    return styles

print(comparison_table.to_string(index=False, float_format=lambda value: f'{value:.3f}'))
display(comparison_table.style.format({
    'Logistic Regression': '{:.3f}',
    'Random Forest': '{:.3f}',
}).apply(highlight_winner, axis=1))

### 3c. Confusion Matrix

The confusion matrix shows where the Random Forest makes correct and incorrect food desert classifications.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not food desert', 'Food desert'],
)
disp.plot(cmap='Greens', values_format='d', ax=ax, colorbar=True)
ax.set_title('Random Forest Confusion Matrix')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
for label in ['TN: correctly not food desert', 'FP: incorrectly food desert', 'FN: missed food desert', 'TP: correctly food desert']:
    ax.plot([], [], linestyle='none', label=label)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=True)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '04_confusion_matrix_rf.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4: SHAP Feature Importance

SHAP values explain how each feature contributes to Random Forest predictions. TreeExplainer is used because it is designed for tree-based models.

### 4a. Global Feature Importance

The beeswarm plot summarizes both feature importance and direction for the food desert class.

In [ ]:
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

def food_desert_shap_values(values):
    if isinstance(values, list):
        return values[1]
    if values.ndim == 3:
        return values[:, :, 1]
    return values

shap_food_test = food_desert_shap_values(shap_values)

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_food_test, X_test, feature_names=MODEL_FEATURES, show=False)
fig = plt.gcf()
ax = plt.gca()
ax.set_title('Random Forest SHAP Summary for Food Desert Class')
ax.set_xlabel('SHAP value impact on food desert prediction')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '04_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### 4b. Top Feature Check (Fairness Preview)

Mean absolute SHAP values identify the strongest model signals. Demographic variables in the top three will be examined more closely in the fairness audit.

In [ ]:
mean_abs_shap = np.abs(shap_food_test).mean(axis=0)
shap_importance = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(shap_importance.to_string(index=False, float_format=lambda value: f'{value:.6f}'))

top_three_features = shap_importance.head(3)['feature'].tolist()
fairness_preview_features = ['pct_nhblack10', 'pct_hisp10']
fairness_features_in_top_three = [feature for feature in fairness_preview_features if feature in top_three_features]

print(f'\nTop 3 SHAP features: {top_three_features}')
if fairness_features_in_top_three:
    print(
        f'Fairness preview: {fairness_features_in_top_three} are in the top 3 features by mean absolute SHAP value, '
        "so demographic composition is among the model\'s strongest signals and should be audited in Week 6."
    )
else:
    print(
        'Fairness preview: pct_nhblack10 and pct_hisp10 are not in the top 3 features by mean absolute SHAP value, '
        'but demographic impacts should still be checked in Week 6.'
    )

### 4c. SHAP Bar Chart

The bar chart separates demographic variables from other predictors to make the fairness preview easier to inspect.

In [ ]:
shap_bar_data = shap_importance.sort_values('mean_abs_shap', ascending=True)
bar_colors = np.where(shap_bar_data['feature'].isin(DEMOGRAPHIC_FEATURES), '#E76F51', '#2A9D8F')

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(shap_bar_data['feature'], shap_bar_data['mean_abs_shap'], color=bar_colors)
ax.set_title('Random Forest Mean Absolute SHAP Values')
ax.set_xlabel('Mean |SHAP value|')
ax.set_ylabel('Feature')
ax.legend(
    handles=[
        Patch(facecolor='#E76F51', label='Demographic feature'),
        Patch(facecolor='#2A9D8F', label='Other feature'),
    ],
    loc='best',
)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '04_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 5: Export Model and Pre-Computed SHAP Values

The Streamlit app will load pre-computed predictions and explanations rather than running live inference. Full-dataset SHAP computation may take about 2-5 minutes depending on the environment.

In [ ]:
full_data = pd.read_csv(DATA_PATH)
fips_col = next((col for col in FIPS_CANDIDATES if col in full_data.columns), None)
if fips_col is None:
    raise KeyError(f'No FIPS identifier column found. Tried: {FIPS_CANDIDATES}')

missing_full_columns = [col for col in [fips_col, TARGET_COL] + MODEL_FEATURES if col not in full_data.columns]
if missing_full_columns:
    raise KeyError(f'Missing required columns for export: {missing_full_columns}')

X_full = full_data[MODEL_FEATURES]
full_predictions = best_rf.predict(X_full)
full_probabilities = best_rf.predict_proba(X_full)[:, 1]

background_size = min(500, len(X_train))
background_sample = shap.sample(X_train, background_size, random_state=42)
full_explainer = shap.TreeExplainer(best_rf, data=background_sample)

start_time = time.perf_counter()
shap_values_full = full_explainer.shap_values(X_full)
full_shap_time = time.perf_counter() - start_time
shap_food_full = food_desert_shap_values(shap_values_full)

feature_names = np.array(MODEL_FEATURES)
top_feature_indices = np.argsort(np.abs(shap_food_full), axis=1)[:, ::-1][:, :3]
top_feature_names = feature_names[top_feature_indices]
top_shap_values = np.take_along_axis(shap_food_full, top_feature_indices, axis=1)

tract_predictions_shap = pd.DataFrame({
    fips_col: full_data[fips_col],
    'predicted_label': full_predictions,
    'prediction_probability': full_probabilities,
    'top1_feature': top_feature_names[:, 0],
    'top1_shap': top_shap_values[:, 0],
    'top2_feature': top_feature_names[:, 1],
    'top2_shap': top_shap_values[:, 1],
    'top3_feature': top_feature_names[:, 2],
    'top3_shap': top_shap_values[:, 2],
    'pct_nhblack10': full_data['pct_nhblack10'],
    'pct_hisp10': full_data['pct_hisp10'],
    'pct_nhwhite10': full_data['pct_nhwhite10'],
    TARGET_COL: full_data[TARGET_COL],
})

parquet_path = OUTPUT_DIR / 'tract_predictions_shap.parquet'
tract_predictions_shap.to_parquet(parquet_path, index=False)

file_size_mb = parquet_path.stat().st_size / (1024 * 1024)
print(f'Full-dataset SHAP time: {full_shap_time:.3f} seconds')
print(f'Saved rows: {tract_predictions_shap.shape[0]:,}')
print(f'Saved columns: {tract_predictions_shap.shape[1]:,}')
print(f'File size: {file_size_mb:.2f} MB')
print(f'Output path: {parquet_path}')

## Section 6: RF Metrics Summary

The final Random Forest metrics are saved for the evaluation notebook and printed alongside the Logistic Regression baseline.

In [ ]:
with open(OUTPUT_DIR / 'rf_metrics.json', 'w') as metrics_file:
    json.dump({key: float(value) for key, value in rf_metrics.items()}, metrics_file, indent=2)

print(
    f"Logistic Regression F1: {baseline_metrics['F1 (food desert class)']:.3f} | "
    f"Random Forest F1: {rf_metrics['F1 (food desert class)']:.3f}"
)
print(
    f"Logistic Regression ROC-AUC: {baseline_metrics['ROC-AUC']:.3f} | "
    f"Random Forest ROC-AUC: {rf_metrics['ROC-AUC']:.3f}"
)
print('Random Forest complete.')
print(
    f"RF F1 (food desert): {rf_metrics['F1 (food desert class)']:.3f} | "
    f"RF ROC-AUC: {rf_metrics['ROC-AUC']:.3f}"
)
print('SHAP pre-computation saved to outputs/tract_predictions_shap.parquet')
print('Next: notebooks/05_evaluation.ipynb')